### Research Question 2: False-Positive Subgroup Investigation

**Are false positives disproportionately associated with particular patient or image characteristics, especially AP/PA view, age, sex, and image acquisition?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [3]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run xraymodel.ipynb

In [4]:
FP_THRESHOLD = 0.7
MIN_GROUP_SIZE = 20  # suppress unstable rates from tiny subgroups

# ---- 1. Align metadata with the images in the current dataset ----
meta = XRAY_DATASET.csv.copy().reset_index(drop=True)
n_images = len(XRAY_DATASET)
if len(meta) != n_images:
    raise ValueError(
        f"Metadata has {len(meta)} rows but dataset has {n_images} images. "
        "Check SubsetDataset metadata alignment before continuing."
    )
if len(XRAY_LABELS_LIST) != n_images or len(MODEL_SCORE_PREDICTIONS_CONVERTED) != n_images:
    raise ValueError("Images, labels and predictions are not aligned in length.")

# Normalize common metadata fields without assuming every source has them.
def _first_existing(columns):
    for c in columns:
        if c in meta.columns:
            return c
    return None

view_col = _first_existing(["AP/PA", "ViewPosition", "View Position", "view"])
sex_col = _first_existing(["Sex", "Patient Gender", "sex", "Gender"])
age_col = _first_existing(["Age", "Patient Age", "age"])

if view_col:
    meta["View"] = meta[view_col].astype(str).str.strip().str.upper()
    meta["View"] = meta["View"].replace({"NAN": "Unknown", "": "Unknown"})
    meta.loc[~meta["View"].isin(["AP", "PA"]), "View"] = "Other/Unknown"
else:
    meta["View"] = "Unavailable"

if sex_col:
    meta["Sex"] = meta[sex_col].astype(str).str.strip().str.upper().replace({
        "M": "Male", "MALE": "Male", "F": "Female", "FEMALE": "Female",
        "NAN": "Unknown", "": "Unknown"
    })
    meta.loc[~meta["Sex"].isin(["Male", "Female"]), "Sex"] = "Unknown"
else:
    meta["Sex"] = "Unavailable"

if age_col:
    meta["Age_years"] = pd.to_numeric(meta[age_col], errors="coerce")
    # Keep plausible adult/pediatric ages; do not silently treat sentinel values as real ages.
    meta.loc[(meta["Age_years"] < 0) | (meta["Age_years"] > 120), "Age_years"] = np.nan
    meta["Age_group"] = pd.cut(
        meta["Age_years"], bins=[0, 40, 50, 60, 70, 80, 120],
        labels=["≤40", "41–50", "51–60", "61–70", "71–80", ">80"],
        include_lowest=True
    ).astype("object").fillna("Unknown")
else:
    meta["Age_years"] = np.nan
    meta["Age_group"] = "Unavailable"

# Candidate acquisition / image metadata fields present in this dataset.
acquisition_candidates = [
    "Frontal/Lateral", "ViewPosition", "AP/PA", "PatientOrientation",
    "ImageType", "PhotometricInterpretation", "Manufacturer",
    "Institution", "Portable", "AcquisitionDeviceProcessingDescription"
]
acquisition_columns = [c for c in acquisition_candidates if c in meta.columns]
print("Metadata columns available:", list(meta.columns))
print("Acquisition-related columns found:", acquisition_columns or "None beyond view metadata")
print("View counts:\n", meta["View"].value_counts(dropna=False))
print("Sex counts:\n", meta["Sex"].value_counts(dropna=False))

# ---- 2. Build long-form outcome table; exclude uncertain/missing labels ----
# IMPORTANT: CheXpert-style labels may contain -1 (uncertain). They are excluded
# from binary FP/FN calculations instead of being silently treated as negatives.
rows = []
for j, pathology in enumerate(DATASET_PATHOLOGIES):
    truth = np.asarray(XRAY_LABELS_LIST[:, j], dtype=float)
    score = np.asarray(MODEL_SCORE_PREDICTIONS_CONVERTED[:, j], dtype=float)
    valid = np.isfinite(truth) & np.isfinite(score) & np.isin(truth, [0, 1])
    pred = (score >= FP_THRESHOLD).astype(int)
    outcome = np.where((truth == 0) & (pred == 1), "False Positive",
             np.where((truth == 1) & (pred == 0), "False Negative",
             np.where((truth == 1) & (pred == 1), "True Positive", "True Negative")))
    tmp = meta.copy()
    tmp["Pathology"] = pathology
    tmp["y_true"] = truth
    tmp["score"] = score
    tmp["y_pred"] = pred
    tmp["Outcome"] = outcome
    tmp["Valid_label"] = valid
    rows.append(tmp.loc[valid])

experiment_df = pd.concat(rows, ignore_index=True)
print(f"Valid binary image-pathology observations: {len(experiment_df):,}")
total_possible = n_images * len(DATASET_PATHOLOGIES)
print(f"Excluded uncertain/missing observations: {total_possible - len(experiment_df):,}")

# Wilson 95% confidence interval for a binomial rate.
def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return np.nan, np.nan
    p = successes / total
    den = 1 + z*z/total
    center = (p + z*z/(2*total)) / den
    half = z * np.sqrt((p*(1-p)/total) + (z*z/(4*total*total))) / den
    return max(0, center-half), min(1, center+half)


def subgroup_metrics(df, group_col, min_n=MIN_GROUP_SIZE):
    """FP rate = FP / all actually-negative cases in subgroup; FNR similarly."""
    records = []
    for (pathology, group), g in df.groupby(["Pathology", group_col], observed=True, dropna=False):
        negatives = g[g["y_true"] == 0]
        positives = g[g["y_true"] == 1]
        n_neg, fp = len(negatives), int((negatives["y_pred"] == 1).sum())
        n_pos, fn = len(positives), int((positives["y_pred"] == 0).sum())
        lo, hi = wilson_interval(fp, n_neg)
        records.append({
            "Pathology": pathology, "Group": str(group),
            "N_images": len(g), "N_actual_negative": n_neg,
            "FP": fp, "FP_rate": fp/n_neg if n_neg else np.nan,
            "FP_CI_low": lo, "FP_CI_high": hi,
            "N_actual_positive": n_pos, "FN": fn,
            "FN_rate": fn/n_pos if n_pos else np.nan,
            "Precision": ((g["y_pred"].eq(1) & g["y_true"].eq(1)).sum() /
                          g["y_pred"].eq(1).sum()) if g["y_pred"].eq(1).sum() else np.nan
        })
    result = pd.DataFrame(records)
    result["Enough_data"] = result["N_actual_negative"] >= min_n
    return result

# ---- PLOT 1–3: FP rate by view, sex, and age group across pathologies ----
def plot_fp_rate_by_subgroup(group_col, min_n=MIN_GROUP_SIZE):
    stats = subgroup_metrics(experiment_df, group_col, min_n=min_n)
    plot_data = stats[stats["Enough_data"] & stats["FP_rate"].notna()].copy()
    if plot_data.empty:
        print(f"No groups for {group_col} have at least {min_n} actual-negative cases.")
        return stats
    fig = px.bar(
        plot_data, x="Pathology", y="FP_rate", color="Group", barmode="group",
        error_y=(plot_data["FP_CI_high"] - plot_data["FP_rate"]),
        error_y_minus=(plot_data["FP_rate"] - plot_data["FP_CI_low"]),
        hover_data=["N_images", "N_actual_negative", "FP", "FP_CI_low", "FP_CI_high"],
        title=f"False-positive rate by {group_col} (threshold={FP_THRESHOLD}; minimum n={min_n})",
        labels={"FP_rate": "False-positive rate among actual negatives", "Pathology": "Finding"}
    )
    fig.update_yaxes(tickformat=".0%", range=[0, 1])
    fig.update_layout(template="plotly_white", height=600, width=1250)
    fig.show()
    return stats

view_stats = plot_fp_rate_by_subgroup("View")
sex_stats = plot_fp_rate_by_subgroup("Sex")
age_stats = plot_fp_rate_by_subgroup("Age_group")

# Also plot other acquisition metadata when it exists and has manageable cardinality.
# This commonly includes frontal/lateral; AP/PA is already covered above.
acquisition_stats = {}
for _col in acquisition_columns:
    if _col in [view_col, "AP/PA", "ViewPosition"]:
        continue
    _n_levels = meta[_col].astype(str).nunique(dropna=False)
    if 2 <= _n_levels <= 12:
        meta[_col + "_group"] = meta[_col].astype(str).replace({"nan": "Unknown", "": "Unknown"})
        # The long table already carries original metadata columns; normalize on that table.
        experiment_df[_col + "_group"] = experiment_df[_col].astype(str).replace({"nan": "Unknown", "": "Unknown"})
        acquisition_stats[_col] = plot_fp_rate_by_subgroup(_col + "_group")

# ---- PLOT 4: AP/PA × sex heatmap for a selected pathology ----
def plot_view_sex_heatmap(pathology):
    d = experiment_df[(experiment_df["Pathology"] == pathology) &
                      (experiment_df["View"].isin(["AP", "PA"])) &
                      (experiment_df["Sex"].isin(["Male", "Female"]))]
    agg = d[d["y_true"] == 0].groupby(["View", "Sex"], observed=True).agg(
        FP=("y_pred", lambda s: int((s == 1).sum())),
        N_actual_negative=("y_pred", "size")
    ).reset_index()
    agg["FP_rate"] = agg["FP"] / agg["N_actual_negative"]
    agg["Label"] = agg.apply(lambda r: f"{r.FP_rate:.1%}<br>FP={r.FP}/{r.N_actual_negative}", axis=1)
    matrix = agg.pivot(index="View", columns="Sex", values="FP_rate").reindex(
        index=["AP", "PA"], columns=["Female", "Male"])
    labels = agg.pivot(index="View", columns="Sex", values="Label").reindex(
        index=["AP", "PA"], columns=["Female", "Male"])
    fig = go.Figure(go.Heatmap(
        z=matrix.values, x=matrix.columns, y=matrix.index, zmin=0, zmax=1,
        colorscale="Reds", colorbar=dict(title="FP rate"),
        text=labels.values, texttemplate="%{text}",
        hovertemplate="View: %{y}<br>Sex: %{x}<br>FP rate: %{z:.1%}<extra></extra>"
    ))
    fig.update_layout(title=f"{pathology}: false-positive rate by view × sex",
                      xaxis_title="Sex", yaxis_title="Image view", template="plotly_white")
    fig.show()
    return agg

# Example: plot_view_sex_heatmap(DATASET_PATHOLOGIES[0])

# ---- PLOT 5: UMAP showing whether FP points concentrate by subgroup ----
def plot_fp_embedding(pathology, subgroup="View", projection="UMAP", max_points=12000):
    if subgroup not in ["View", "Sex", "Age_group"]:
        raise ValueError("subgroup must be View, Sex, or Age_group")
    label_idx = DATASET_LABEL_INDICES_MAP[pathology]
    truth = np.asarray(XRAY_LABELS_LIST[:, label_idx], dtype=float)
    scores = np.asarray(MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx], dtype=float)
    valid = np.isfinite(truth) & np.isfinite(scores) & np.isin(truth, [0, 1])
    fp_mask = valid & (truth == 0) & (scores >= FP_THRESHOLD)
    if projection == "S-UMAP":
        emb = downprojector.calculate_projection("feature", "S-UMAP",
                    target_labels=XRAY_LABELS_LIST, label_name=pathology)
    else:
        emb = downprojector.calculate_projection("feature", projection)
    d = meta.copy()
    d["x"] = emb[:, 0]; d["y"] = emb[:, 1]
    d["is_fp"] = fp_mask
    d["score"] = scores
    # Plot all valid cases faintly, then highlight FP cases coloured by subgroup.
    keep = valid
    if keep.sum() > max_points:
        rng = np.random.default_rng(RANDOM_STATE)
        keep_idx = rng.choice(np.flatnonzero(keep), max_points, replace=False)
        keep = np.zeros(len(d), dtype=bool); keep[keep_idx] = True
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=d.loc[keep & valid, "x"], y=d.loc[keep & valid, "y"], mode="markers",
        name="All valid cases", marker=dict(color="lightgrey", size=4, opacity=0.25),
        hoverinfo="skip"
    ))
    for group in sorted(d.loc[fp_mask, subgroup].astype(str).unique()):
        m = fp_mask & d[subgroup].astype(str).eq(group).to_numpy()
        fig.add_trace(go.Scattergl(
            x=d.loc[m, "x"], y=d.loc[m, "y"], mode="markers", name=f"FP · {group}",
            marker=dict(size=6, opacity=0.85),
            customdata=np.column_stack([d.loc[m, "score"], d.loc[m, "Age_years"]]),
            hovertemplate=f"{subgroup}: {group}<br>Score: %{{customdata[0]:.3f}}<br>Age: %{{customdata[1]}}<extra></extra>"
        ))
    fig.update_layout(title=f"{pathology}: false positives in {projection} coloured by {subgroup}",
                      xaxis_title=f"{projection} 1", yaxis_title=f"{projection} 2",
                      template="plotly_white", width=1000, height=750)
    fig.show()
    return fig

# Examples:
# plot_fp_embedding("Cardiomegaly", subgroup="View")
# plot_fp_embedding("Cardiomegaly", subgroup="Sex")
# plot_fp_embedding("Cardiomegaly", subgroup="Age_group")

# ---- PLOT 6: Score distributions among actual negatives by subgroup ----
def plot_negative_score_distributions(pathology, subgroup="View"):
    d = experiment_df[(experiment_df["Pathology"] == pathology) & (experiment_df["y_true"] == 0)].copy()
    d = d[d[subgroup].notna()]
    fig = px.violin(d, x=subgroup, y="score", color=subgroup, box=True, points="outliers",
                    title=f"{pathology}: model scores on actual negatives by {subgroup}",
                    labels={"score": "Model score", subgroup: subgroup})
    fig.add_hline(y=FP_THRESHOLD, line_dash="dash", line_color="black",
                  annotation_text=f"Decision threshold = {FP_THRESHOLD}")
    fig.update_yaxes(range=[0, 1])
    fig.update_layout(template="plotly_white", showlegend=False, height=550, width=900)
    fig.show()
    return d

# Example: plot_negative_score_distributions("Cardiomegaly", "View")

# ---- PLOT 7: rank subgroup disparities; export the underlying tables ----
all_subgroup_stats = pd.concat([
    view_stats.assign(Metadata="View"),
    sex_stats.assign(Metadata="Sex"),
    age_stats.assign(Metadata="Age group"),
    *[v.assign(Metadata=k) for k, v in acquisition_stats.items()]
], ignore_index=True)
all_subgroup_stats.to_csv("false_positive_subgroup_metrics.csv", index=False)

# Summary of available metadata and a first cross-pathology comparison.
print("\nLargest view-related FP-rate differences (only groups with adequate n):")
view_valid = view_stats[view_stats["Enough_data"]].copy()
view_spread = view_valid.pivot(index="Pathology", columns="Group", values="FP_rate")
if {"AP", "PA"}.issubset(view_spread.columns):
    view_spread["AP_minus_PA"] = view_spread["AP"] - view_spread["PA"]
    display(view_spread.sort_values("AP_minus_PA", key=lambda s: s.abs(), ascending=False))
else:
    print("Both AP and PA groups with sufficient sample sizes were not available for every pathology.")

print("\nSaved: false_positive_subgroup_metrics.csv")


Metadata columns available: ['PatientID', 'Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'view', 'patientid', 'age_years', 'sex_male', 'sex_female', 'View', 'Age_years', 'Age_group']
Acquisition-related columns found: ['Frontal/Lateral', 'AP/PA']
View counts:
 View
PA    5663
AP    4337
Name: count, dtype: int64
Sex counts:
 Sex
Male      5800
Female    4200
Name: count, dtype: int64
Valid binary image-pathology observations: 110,000
Excluded uncertain/missing observations: 0



Largest view-related FP-rate differences (only groups with adequate n):


Group,AP,PA,AP_minus_PA
Pathology,,,
Atelectasis,0.530774,0.256583,0.274190
Edema,0.369931,0.113813,0.256118
Effusion,0.337228,0.118652,0.218576
Cardiomegaly,0.341157,0.141418,0.199739
Lung Opacity,0.411059,0.223024,0.188035
Consolidation,0.245316,0.121704,0.123612
Enlarged Cardiomediastinum,0.200867,0.109084,0.091783
Lung Lesion,0.093886,0.146830,-0.052944
Pneumonia,0.118367,0.127979,-0.009612



Saved: false_positive_subgroup_metrics.csv


### Answering the Questions

**False-positive rate: AP vs PA**: The diagram shows that the false-positive rate for AP is overall higher than for PA. The only exception is Lung lesion where PA is higher.

**False-positive rate: male vs female**: The diagrams does not show significant differences between male and female. 

**False-positive rate: age**: The diagrams shows that in particular the age group **>80** has a high false-positive rate.